# Deutsch–Jozsa audit: constant or balanced?

**Goal:** Given a promised 4-bit Boolean function $f:\{0,1\}^4\to\{0,1\}$, decide whether it is constant or balanced. This is an oracle-classification demonstration, **not** a hardware random-number generator and it does not generate random numbers.

There are 16 possible inputs. A balanced function returns 1 on exactly 8 of them. Under the promise, Deutsch–Jozsa distinguishes the two cases with one oracle query (ideal circuit).

## 1. Setup (Google Colab)

Run the installation cell, then run the notebook from top to bottom. Qiskit Aer simulates the circuits on a classical computer. The fixed seed makes the shot sampling reproducible; it does not create or represent a physical random-number generator.


In [ ]:
%pip -q install qiskit qiskit-aer matplotlib

In [ ]:
import qiskit, qiskit_aer, matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
print("Qiskit", qiskit.__version__)
print("Aer", qiskit_aer.__version__)

## 2. Oracle promise and examples

The oracle must be reversible, so it is represented as $U_f|x,y\rangle=|x,y\oplus f(x)\rangle$. Here $x$ is a 4-bit input and $y$ is one auxiliary bit. The input wires are `q0`–`q3`; `q4` is the auxiliary.

We prepare the auxiliary as $|1\rangle$ and apply Hadamards to all five qubits. This makes its state $| - \rangle=(|0\rangle-|1\rangle)/\sqrt2$. Flipping that auxiliary conditionally on $f(x)$ gives **phase kickback**: each input basis state acquires a sign $(-1)^{f(x)}$.

We implement three oracle examples:

- **Constant zero:** no oracle gate, so $f(x)=0$ for all 16 inputs.
- **Constant one:** flip the auxiliary, so $f(x)=1$ for all 16 inputs.
- **Balanced:** CNOT from input `q0` to auxiliary `q4`. This is $f(x)=x_0$, which equals 1 for exactly 8 of the 16 inputs.

The balanced oracle is deliberately simple and structured, making its behavior easy to explain. It is a valid balanced function under the promise; it does not model a general RNG.


In [ ]:
N_INPUTS = 4
AUX = 4

def make_dj_circuit(kind: str, shots: int = 1024) -> QuantumCircuit:
    """Build Deutsch–Jozsa circuit. kind: constant0, constant1, or balanced."""
    if kind not in {"constant0", "constant1", "balanced"}:
        raise ValueError("kind must be constant0, constant1, or balanced")
    qc = QuantumCircuit(N_INPUTS + 1, N_INPUTS)
    qc.x(AUX)
    qc.h(range(N_INPUTS + 1))
    qc.barrier(label="oracle begins")
    if kind == "constant1":
        qc.x(AUX)
    elif kind == "balanced":
        qc.cx(0, AUX)  # f(x0,x1,x2,x3) = x0
    qc.barrier(label="oracle ends")
    qc.h(range(N_INPUTS))
    qc.measure(range(N_INPUTS), range(N_INPUTS))
    return qc

for kind in ("constant0", "constant1", "balanced"):
    print("\n", kind)
    display(make_dj_circuit(kind).draw("mpl"))

## 3. Simulate and inspect counts

Only the four input qubits are measured; the auxiliary qubit is not. The DJ decision rule under the promise is: **all-zero input outcome means constant; any nonzero input outcome means balanced.** In the ideal circuit, constant gives all-zero with certainty, while balanced gives all-zero with probability zero. In Qiskit, the displayed bit string is ordered `c3 c2 c1 c0` (highest-index classical bit first); that ordering does not affect the zero-versus-nonzero test.


In [ ]:
simulator = AerSimulator()
SHOTS = 1024
SEED = 2026
results = {}
for kind in ("constant0", "constant1", "balanced"):
    circuit = make_dj_circuit(kind)
    compiled = transpile(circuit, simulator, seed_transpiler=SEED)
    counts = simulator.run(compiled, shots=SHOTS, seed_simulator=SEED).result().get_counts()
    results[kind] = counts
    print(f"{kind}: {counts}")

## 4. Histograms and decision rule

The histogram shows the outcomes sampled over repeated simulator shots. The selected balanced function has a deterministic nonzero answer, so its histogram should contain only `0001`; a different balanced oracle can produce a different nonzero pattern. The promise lets the algorithm distinguish the cases after one ideal query. We use 1,024 shots here so the counts are easy to display.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (kind, counts) in zip(axes, results.items()):
    keys = sorted(counts)
    ax.bar(keys, [counts[k] for k in keys], color="#4f6bed")
    ax.set_title(kind.replace("constant", "constant "))
    ax.set_xlabel("4-bit input measurement")
    ax.set_ylabel("shots")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

def classify(counts):
    """Apply the promise-based DJ rule to measured input counts."""
    observed_zero = counts.get("0000", 0) > 0
    return "constant" if observed_zero else "balanced"

for kind, counts in results.items():
    print(f"{kind:10s} -> classified as {classify(counts)}; all-zero shots = {counts.get('0000', 0)}")


## 5. Result analysis and limitations

- **Constant:** The phase is the same for every input. After the final Hadamards, the 16 contributions add constructively at $|0000\rangle$. Constant-one adds a global phase $-1$, which does not change measurement probabilities.
- **Balanced:** Half the inputs have phase $+1$ and half have phase $-1$. Their contributions to the all-zero amplitude cancel: $\langle 0^4|H^{\otimes4}\sum_x(-1)^{f(x)}|x\rangle=2^{-4}\sum_x(-1)^{f(x)}=0$. Thus a nonzero measurement is guaranteed in the ideal circuit. For our $f(x)=x_0$ oracle, the output is deterministically `0001` in Qiskit's displayed bit order.
- **Why only one query?** The initial Hadamards create a superposition over all 16 inputs. The oracle encodes function values as phases in one query, and the final Hadamards reveal whether the phases all agree or cancel at zero.
- This conclusion depends on the promise that the function is constant or balanced. If a function satisfies neither case, this decision rule can give a misleading label.
- On noisy hardware, readout or gate errors can produce exceptions (such as a nonzero result for a constant function). The ideal simulator used here omits hardware noise.
- This experiment audits a deliberately constructed Boolean oracle. It does not assess entropy, unpredictability, bias, or security of an RNG, and it does not generate random numbers.

### Presentation takeaway
Deutsch–Jozsa demonstrates a one-query quantum advantage for a tightly promised classification problem. It is an oracle audit, not a random-number-generation method.
